# 06 Rolling OLS and Ridge Regression

This notebook adds dynamic hedge-ratio estimation to the triangular statistical arbitrage research engine.

The research question here is whether the residual changes materially when the hedge basket is estimated with static initial-window OLS, rolling OLS, or rolling ridge regression.

The project remains a reproducible research and backtesting system. The outputs below should not be interpreted as live trading signals.

## Concept map

A static hedge ratio uses one coefficient set for the sample. That can become stale when sector exposure, volatility, correlation, or single-name news changes.

A rolling window estimates coefficients from the most recent observations before the prediction date. For date `t`, the training window ends at `t - 1`. This avoids look-ahead bias.

Ridge regression changes the objective from pure squared-error minimization to squared-error plus a coefficient-size penalty. This can stabilize hedge ratios when the two hedge assets are correlated.

The target residual is still:

```text
residual_t = log(P_A,t) - [alpha_t + beta_1,t log(P_B,t) + beta_2,t log(P_C,t)]
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DEFAULT_DATABASE_PATH,
    DEFAULT_RIDGE_ALPHA,
    DEFAULT_ROLLING_WINDOW,
    FIGURES_DIR,
    PROCESSED_DATA_DIR,
    SQL_DIR,
    TRIPLET_DEFINITIONS,
)
from src.database import (
    connect_database,
    initialize_database,
    store_dynamic_residuals,
    store_residual_method_summary,
    store_ridge_coefficients,
    store_rolling_coefficients,
)
from src.regression import estimate_dynamic_hedges_for_triplets
from src.residuals import compare_residual_stability

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)

## Data loading

The preferred input is the cleaned log-price table produced by the ingestion pipeline. If that table is not available in the local SQLite database, this notebook uses a clearly marked synthetic placeholder dataset so the code path can still be inspected and tested. Placeholder outputs should be overwritten with actual project data before any interpretation.

In [ ]:
def load_log_prices_from_database(database_path):
    if not Path(database_path).exists():
        return None
    with connect_database(database_path) as conn:
        candidates = ["log_prices", "clean_log_prices", "price_log_returns"]
        for table in candidates:
            try:
                frame = pd.read_sql_query(f"SELECT * FROM {table}", conn)
            except Exception:
                continue
            if "date" in frame.columns:
                frame["date"] = pd.to_datetime(frame["date"])
                frame = frame.set_index("date")
            numeric = frame.select_dtypes(include=["number"]).copy()
            if not numeric.empty:
                return numeric
    return None


def make_placeholder_log_prices(n_obs=260, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=n_obs)
    market = np.cumsum(rng.normal(0.0005, 0.01, n_obs))
    tech = market + np.cumsum(rng.normal(0.0002, 0.006, n_obs))
    energy = np.cumsum(rng.normal(0.0004, 0.012, n_obs))
    financials = market * 0.55 + np.cumsum(rng.normal(0.0003, 0.008, n_obs))

    data = {
        "QQQ": 5.6 + market + 0.65 * tech,
        "XLK": 5.2 + market + 0.85 * tech,
        "SMH": 4.8 + market + 1.10 * tech,
        "XLY": 4.9 + 0.80 * market + 0.35 * tech,
        "XLF": 4.4 + financials,
        "KRE": 3.8 + 1.25 * financials + np.cumsum(rng.normal(0.0, 0.006, n_obs)),
        "XLE": 4.5 + energy,
        "USO": 3.9 + 1.15 * energy + np.cumsum(rng.normal(0.0, 0.01, n_obs)),
    }
    frame = pd.DataFrame(data, index=dates)
    frame["NVDA"] = 0.20 + 0.55 * frame["SMH"] + 0.45 * frame["QQQ"] + rng.normal(0, 0.015, n_obs)
    frame["AMD"] = 0.10 + 0.65 * frame["SMH"] + 0.35 * frame["QQQ"] + rng.normal(0, 0.018, n_obs)
    frame["AAPL"] = 0.15 + 0.60 * frame["QQQ"] + 0.40 * frame["XLK"] + rng.normal(0, 0.012, n_obs)
    frame["MSFT"] = 0.22 + 0.50 * frame["QQQ"] + 0.50 * frame["XLK"] + rng.normal(0, 0.011, n_obs)
    frame["JPM"] = 0.18 + 0.70 * frame["XLF"] + 0.30 * frame["KRE"] + rng.normal(0, 0.013, n_obs)
    frame["BAC"] = 0.08 + 0.75 * frame["XLF"] + 0.25 * frame["KRE"] + rng.normal(0, 0.014, n_obs)
    frame["XOM"] = 0.16 + 0.80 * frame["XLE"] + 0.20 * frame["USO"] + rng.normal(0, 0.012, n_obs)
    frame["CVX"] = 0.14 + 0.78 * frame["XLE"] + 0.22 * frame["USO"] + rng.normal(0, 0.012, n_obs)
    frame["TSLA"] = 0.05 + 0.55 * frame["QQQ"] + 0.45 * frame["XLY"] + rng.normal(0, 0.025, n_obs)
    frame["AMZN"] = 0.12 + 0.58 * frame["QQQ"] + 0.42 * frame["XLY"] + rng.normal(0, 0.014, n_obs)
    return frame

log_prices = load_log_prices_from_database(DEFAULT_DATABASE_PATH)
using_placeholder_data = log_prices is None
if using_placeholder_data:
    log_prices = make_placeholder_log_prices()

log_prices.shape, using_placeholder_data

## Estimation design

The rolling estimators use a trailing window. The fitted value for each date is out-of-sample relative to the coefficient estimate for that date.

The comparison table uses the same evaluation dates across methods:

- static OLS fitted on the initial window
- rolling OLS fitted on the trailing window
- rolling ridge fitted on the trailing window

In [ ]:
window = min(DEFAULT_ROLLING_WINDOW, max(30, len(log_prices) // 4))
ridge_alpha = DEFAULT_RIDGE_ALPHA

available_symbols = set(log_prices.columns)
active_triplets = [
    triplet for triplet in TRIPLET_DEFINITIONS
    if {triplet["target"], triplet["hedge_1"], triplet["hedge_2"]}.issubset(available_symbols)
]

results = estimate_dynamic_hedges_for_triplets(
    log_prices=log_prices,
    triplets=active_triplets,
    window=window,
    ridge_alpha=ridge_alpha,
)

rolling_coefficients = results["rolling_coefficients"]
ridge_coefficients = results["ridge_coefficients"]
dynamic_residuals = results["dynamic_residuals"]
summary = compare_residual_stability(dynamic_residuals)

rolling_coefficients.head(), ridge_coefficients.head(), summary.head()

## Storage

The database tables below store the coefficient paths, dynamic residuals, and residual-method comparison. The schema uses triplet, date, and method keys so later backtests can select a residual definition directly.

In [ ]:
initialize_database(DEFAULT_DATABASE_PATH, SQL_DIR / "schema.sql")
with connect_database(DEFAULT_DATABASE_PATH) as conn:
    store_rolling_coefficients(conn, rolling_coefficients, if_exists="replace")
    store_ridge_coefficients(conn, ridge_coefficients, if_exists="replace")
    store_dynamic_residuals(conn, dynamic_residuals, if_exists="replace")
    store_residual_method_summary(conn, summary, if_exists="replace")

summary_path = PROCESSED_DATA_DIR / "static_vs_dynamic_residual_summary.csv"
summary.to_csv(summary_path, index=False)
summary_path

## Hedge-ratio stability plot

The figure shows how the two hedge coefficients move through time for one triplet. In the full project, this plot should be reviewed for every triplet before using the residuals in a backtest.

In [ ]:
if not rolling_coefficients.empty and not ridge_coefficients.empty:
    plot_triplet = rolling_coefficients["triplet_id"].iloc[0]
    roll_plot = rolling_coefficients[rolling_coefficients["triplet_id"] == plot_triplet].copy()
    ridge_plot = ridge_coefficients[ridge_coefficients["triplet_id"] == plot_triplet].copy()
    roll_plot["date"] = pd.to_datetime(roll_plot["date"])
    ridge_plot["date"] = pd.to_datetime(ridge_plot["date"])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(roll_plot["date"], roll_plot["beta_1"], label="rolling OLS beta 1")
    ax.plot(roll_plot["date"], roll_plot["beta_2"], label="rolling OLS beta 2")
    ax.plot(ridge_plot["date"], ridge_plot["beta_1"], linestyle="--", label="rolling ridge beta 1")
    ax.plot(ridge_plot["date"], ridge_plot["beta_2"], linestyle="--", label="rolling ridge beta 2")
    ax.set_title(f"Hedge Ratio Stability: {plot_triplet}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Coefficient")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    figure_path = FIGURES_DIR / "hedge_ratio_stability.png"
    fig.savefig(figure_path, dpi=160)
    plt.close(fig)
else:
    figure_path = None

figure_path

## Backtest comparison table handoff

This notebook does not claim that dynamic residuals improve trading performance. The table below is a structured handoff for the backtest engine. It records residual-method diagnostics on the same triplets and dates. A later notebook can replace diagnostic columns with full transaction-cost-aware backtest metrics.

In [ ]:
backtest_comparison = summary.copy()
backtest_comparison["data_source"] = "synthetic_placeholder" if using_placeholder_data else "project_database"
backtest_comparison["interpretation_status"] = np.where(
    using_placeholder_data,
    "placeholder_not_for_research_interpretation",
    "ready_for_backtest_review",
)
backtest_path = PROCESSED_DATA_DIR / "backtest_comparison_table.csv"
backtest_comparison.to_csv(backtest_path, index=False)
backtest_comparison.head()

## Interpretation notes

A smoother ridge coefficient path is not automatically better. Ridge can reduce variance but introduces bias. Rolling OLS can adapt to changing relationships, but a short window can overfit recent noise. Static OLS can be stale, but it is more stable and easier to interpret.

The correct next step is to feed each residual definition into the same baseline backtest rules used here, with the same transaction-cost assumptions and the same no-look-ahead convention.